<a href="https://colab.research.google.com/github/AlexandreLouzada/exercicios-analise-dados/blob/master/dashboard_streamlit_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Desenvolvimento e Publicação de um Dashboard Interativo — GABARITO

**Objetivo:** Painel de BI com filtros, métricas e gráficos, publicado na nuvem.

---


## 🟢 Fase 1 — Estrutura Básica e Otimização de Desempenho

**P1.** `st.title('Dashboard de Vendas')`.
**P2.** Função `carregar_dados()` lê o CSV com Pandas.
**P3.** `@st.cache_data`: o Streamlit reexecuta o script a cada clique — o cache guarda o DataFrame na memória e evita reler o arquivo a cada interação.

In [ ]:
import streamlit as st
import pandas as pd


@st.cache_data
def carregar_dados():
    df = pd.read_csv("vendas.csv", parse_dates=["data"])
    return df


st.title("Dashboard de Vendas")

df = carregar_dados()
st.write("Linhas carregadas:", df.shape[0])


## 🟡 Fase 2 — Layout e Filtros Laterais

**P1.** `st.sidebar.title('Filtros')` — controles globais na lateral.
**P2.** `st.sidebar.multiselect(...)` com as categorias.
**P3.** Regra de ouro: usar o valor do widget para filtrar o DataFrame (`df[df['categoria'].isin(...)]`).

In [ ]:
st.sidebar.title("Filtros")

lista_categorias = df["categoria"].unique().tolist()
categorias_sel = st.sidebar.multiselect(
    "Selecione as Categorias",
    options=lista_categorias,
    default=lista_categorias,
)

df_filtrado = df[df["categoria"].isin(categorias_sel)]
st.write("Após filtro:", df_filtrado.shape[0], "linhas")


## 🔵 Fase 3 — Métricas em Destaque e Visualização

**P1.** `st.columns([1, 1])` divide a tela em colunas proporcionais.
**P2.** `st.metric` para Receita Total e Total de Pedidos.
**P3.** `st.tabs` para navegação.
**P4.** `aba1`: agrupar por mês (`resample('ME')`) e `st.area_chart`.
**P5.** `aba2`: `st.dataframe` (tabela ordenável) + `st.download_button` (CSV).

In [ ]:
col1, col2 = st.columns([1, 1])

with col1:
    receita_total = df_filtrado["receita"].sum()
    st.metric(label="Receita Total", value=f"R$ {receita_total:,.2f}")

with col2:
    total_pedidos = len(df_filtrado)
    st.metric(label="Total de Pedidos", value=f"{total_pedidos:,}")

aba1, aba2 = st.tabs(["Evolução Mensal", "Tabela de Dados"])

with aba1:
    receita_mensal = (
        df_filtrado.set_index("data")["receita"]
        .resample("ME")
        .sum()
        .rename("receita")
    )
    st.area_chart(receita_mensal)

with aba2:
    st.dataframe(df_filtrado, use_container_width=True)

    csv_bytes = df_filtrado.to_csv(index=False).encode("utf-8")
    st.download_button(
        label="Baixar recorte em CSV",
        data=csv_bytes,
        file_name="recorte_vendas.csv",
        mime="text/csv",
    )


## 🚀 Fase 4 — Publicação na Nuvem (Deploy)

O app é um arquivo `.py`, então **não roda dentro do Colab**.

**Local (teste):** `pip install streamlit pandas matplotlib` → `streamlit run meu_dashboard.py`

**Nuvem (Streamlit Community Cloud — share.streamlit.io):**

1. **Versionamento:** crie um repositório no **GitHub** e faça commit de `meu_dashboard.py` e `vendas.csv`.
2. **Dependências:** adicione um `requirements.txt` listando `streamlit`, `pandas`, `matplotlib`.
3. **Conectar:** acesse share.streamlit.io e vincule sua conta GitHub.
4. **Deploy:** selecione o repositório, aponte o arquivo principal `meu_dashboard.py` e clique em **Deploy**.

Em alguns minutos a plataforma gera um **link público** do dashboard.